# Visit Fudan Sightseeing Spots


## Improt packages

In [5]:
!pip install --no-deps "coord_convert==0.2.1"

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/1f/97/6ab0b423924de14329a6a7f231e7712bb2b0c60c5c47efa08d8d65c841fc/coord_convert-0.2.1-py3-none-any.whl (6.3 kB)



[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import pandas as pd
import numpy as np
from tqdm import tqdm

import folium                             ## Package for Plotting on Map
from folium.plugins import BeautifyIcon
#import mpu                                ## Package for checking haversine distances
from haversine import haversine as haversine_distance
import openrouteservice as ors            ## Package for checking driving distances
from coord_convert.transform import wgs2gcj, wgs2bd, gcj2wgs, gcj2bd, bd2wgs, bd2gcj 
import requests,time

from rsome import ro
from rsome import ort_solver as ort

## Read data and visualize locations

In [7]:
## Obtain the geolocation from baidu map.

### 复旦管院   121.513081,31.302164
### 六教      121.511338,31.300683
### 经济学院   121.511185,31.301323
### 曦园      121.51119,31.302858
### 3教       121.510916,31.303645
### 主席像     121.510134,31.303182
### 新闻学院   121.515686,31.305269
### 旦苑      121.512946,31.306387
### 光华楼     121.51189,31.305388
### 光草      121.512353,31.304505
### 智库楼     121.510619,31.304721
### 软件学院   121.510592,31.305469
### 科学楼     121.509276,31.305338
### 微电子楼   121.511149,31.30594
### 逸夫楼     121.508338,31.304648
### 遗传学楼   121.507606,31.30491
### 相辉堂     121.506209,31.303414
### 校史馆     121.50699,31.302133
### 燕园      121.507686,31.302164

# lat = [40.8352, 40.8342, 40.8335, 40.8323, 40.8311, 40.8308, 40.8304, 40.8315, 40.8325, 40.8332, 40.8339, 40.8345,
#            40.8352]
# lon = [114.8886, 114.8883, 114.8881, 114.8877, 114.8873, 114.8888, 114.8902, 114.8909, 114.8916, 114.8919, 114.8922,
#            114.8917, 114.8886]


data = np.array([[121.513081,31.302164],[121.511338,31.300683],[121.511185,31.301323],[121.51119,31.302858],
                [121.510916,31.303645],[121.510134,31.303182],[121.515686,31.305269],[121.512946,31.306387],
                [121.51189,31.305388],[121.512353,31.304505],[121.510619,31.304721],[121.510592,31.305469],
                [121.509276,31.305338],[121.511149,31.30594],[121.508338,31.304648],[121.507606,31.30491],
                [121.506209,31.303414],[121.50699,31.302133],[121.507686,31.302164]])
print(data)

[[121.513081  31.302164]
 [121.511338  31.300683]
 [121.511185  31.301323]
 [121.51119   31.302858]
 [121.510916  31.303645]
 [121.510134  31.303182]
 [121.515686  31.305269]
 [121.512946  31.306387]
 [121.51189   31.305388]
 [121.512353  31.304505]
 [121.510619  31.304721]
 [121.510592  31.305469]
 [121.509276  31.305338]
 [121.511149  31.30594 ]
 [121.508338  31.304648]
 [121.507606  31.30491 ]
 [121.506209  31.303414]
 [121.50699   31.302133]
 [121.507686  31.302164]]


In [8]:
bd_lon = data[:,0]
bd_lat = data[:,1]
N=len(bd_lon)

lon = np.zeros(N)
lat = np.zeros(N)
for i in range(N):
    lon[i], lat[i] = bd2gcj(bd_lon[i], bd_lat[i])    ##$ convert the (lon,lat) coordinate to standard format.


FDmap = folium.Map(location=[lat[0], lon[0]], zoom_start=15,tiles='http://webrd02.is.autonavi.com/appmaptile?lang=zh_cn&size=1&scale=1&style=7&x={x}&y={y}&z={z}',attr='default') #高德地图
for i in range(N):
    folium.Marker(
        location=[lat[i], lon[i]],
        icon=BeautifyIcon(
            icon_shape='marker',
            number=int(i + 1),
            spin=True,
            text_color='red',
            background_color="#FFF",
            inner_icon_style="font-size:12px;padding-top:-5px;"
        )
    ).add_to(FDmap)
FDmap

### Prepare the distance matrix

In [11]:
haversine_distance((lat[0], lon[0]),(lat[1], lon[1]))

0.23292020919915554

In [13]:
distance_mat = np.array([[haversine_distance((lat[i], lon[i]), (lat[j], lon[j]))
                          for j in range(N)] 
                          for i in range(N)])

## Model formulation: the first try

**(Baseline formulation)** Label the centres with the numbers $1,\ldots,N$ and define:
$$
x_{ij} = \begin{cases} 
1 & \text{the path goes from centre } i \text{ to centre } j \\ 
0 & \text{otherwise} 
\end{cases}
$$

For $i=1,\ldots,N$, let $u_i$ be a dummy variable, and finally take $d_{ij} > 0$ to be the distance from centre $i$ to centre $j$. Then TSP can be written as the following integer linear programming problem:
$$
\begin{align}
\min &\sum_{i=1}^n \sum_{j=1}^n d_{ij}x_{ij}\colon &&  \\
     & \sum_{i=1}^N x_{ij} = 1 && j=1, \ldots, N; \\
     & \sum_{j=1}^N x_{ij} = 1 && i=1, \ldots, N; \\
     & x_{ij} \in \{0,1\}  && i,j=1, \ldots, N; \\
     & x_{ii} = 0  && i= 1, \ldots, N; \\
\end{align}
$$

In [19]:
def TSP_baseline(N, distance):
    ## Miller–Tucker–Zemlin formulation
    m = ro.Model('TSP-Baseline')
    x = m.dvar((N,N),'B')
    u = m.dvar(N)
    
    ########## Fill up your code here to complete the optimization model  ##########
    m.min(sum(distance_mat[i, j] * x[i, j]for i in range(N) for j in range(N)))
    for i in range(N):
        m.st(sum(x[i, j] for j in range(N)) == 1)
    for j in range(N):
        m.st(sum(x[i, j] for i in range(N)) == 1)
    for _ in range(N):
        m.st(x[_, _] == 0)
    for i in range(1, N):
        for j in range(1, N):
            m.st(u[i] - u[j] + N * x[i, j] <= N -1)
    m.st(u[0] == 0)
    ################################################################################

    m.solve(ort)
    x_star = x.get()
    obj = m.get()
    return x_star, obj

There are two parameters for this model: number of centers $N$ and distrance matrix $\boldsymbol{d}$. 

The number of centers $N$ is 10 for this problem, but the distance matrix is not known. However, you can approximately obtain it via the `haversine_distance` method of `mpu` package as follows:

In [20]:
x_baseline, obj_baseline = TSP_baseline(N, distance_mat[0:N,0:N])

Being solved by OR-Tools...
Solution status: 0
Running time: 0.8133s


In [21]:
print(x_baseline)

[[0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.

In [22]:
ind = np.argwhere( x_baseline )
print(ind)

[[ 0  9]
 [ 1  0]
 [ 2  1]
 [ 3  2]
 [ 4  3]
 [ 5  4]
 [ 6  7]
 [ 7  8]
 [ 8 13]
 [ 9  6]
 [10 12]
 [11 10]
 [12 14]
 [13 11]
 [14 15]
 [15 16]
 [16 17]
 [17 18]
 [18  5]]


### Obtain the routes given by the optimal solution

In [23]:
## obtain [lon, lat] pairs for the route
def obtain_route(x_star, lon, lat):
    N = len(lon)
    seq_route = -np.ones(N + 1) # route sequence
    lon_route = -np.ones(N + 1)
    lat_route = -np.ones(N + 1)

    # starting point is food center 1
    lon_route[0] = lon[0]
    lat_route[0] = lat[0]
    seq_route[0]= 1

    ind = np.argwhere( x_star )   # find where x=1
    
    ind_j = 0                     # The first stop
    for i in range(N):
        ind_j = ind[ind_j,1]      # Check the next stop
        seq_route[i+1] = ind_j + 1
        lon_route[i+1] = lon[ind_j]
        lat_route[i+1] = lat[ind_j]
        
    return seq_route, lon_route, lat_route

In [24]:
seq_route_tsp, lon_route_tsp, lat_route_tsp = obtain_route(x_baseline, lon[0:N], lat[0:N])
seq_route_tsp

array([ 1., 10.,  7.,  8.,  9., 14., 12., 11., 13., 15., 16., 17., 18.,
       19.,  6.,  5.,  4.,  3.,  2.,  1.])

 **This does not seem to be correct... Is our code wrong? Is our model wrong?**

## Model formulation: Miller–Tucker–Zemlin formulation


**(Miller–Tucker–Zemlin formulation)** Label the centres with the numbers $1,\ldots,N$ and define:
$$
x_{ij} = \begin{cases} 
1 & \text{the path goes from centre } i \text{ to centre } j \\ 
0 & \text{otherwise} 
\end{cases}
$$

For $i=1,\ldots,N$, let $u_i$ be a dummy variable, and finally take $d_{ij} > 0$ to be the distance from centre $i$ to centre $j$. Then TSP can be written as the following integer linear programming problem:
$$
\begin{align}
\min &\sum_{i=1}^n \sum_{j=1}^n d_{ij}x_{ij}\colon &&  \\
     & \sum_{i=1}^N x_{ij} = 1 && j=1, \ldots, N; \\
     & \sum_{j=1}^N x_{ij} = 1 && i=1, \ldots, N; \\
     & u_i-u_j +Nx_{ij} \le N-1 && i,j=2,\ldots,N;  \\
     & 1 \le u_i \le N-1 && 2 \le i \le N; \\
     & u_{i} \in \mathbf{Z} && i=2, \ldots, N; \\
     & x_{ij} \in \{0,1\}  && i,j=1, \ldots, N; \\
     & x_{ii} = 0  && i= 1, \ldots, N; \\
\end{align}
$$

In [25]:
def TSP_MTZ(N, distance):
    ## Miller–Tucker–Zemlin formulation
    m = ro.Model('TSP-MTZ')
    x = m.dvar((N,N),'B')
    u = m.dvar(N)

    ########## Fill up your code here to complete the optimization model  ##########
    m.min(sum(distance_mat[i, j] * x[i, j]for i in range(N) for j in range(N)))
    for i in range(N):
        m.st(sum(x[i, j] for j in range(N)) == 1)
    for j in range(N):
        m.st(sum(x[i, j] for i in range(N)) == 1)
    for _ in range(N):
        m.st(x[_, _] == 0)
    for i in range(1, N):
        m.st(1 <= u[i] <= N)
        for j in range(1, N):
            m.st(u[i] - u[j] + N * x[i, j] <= N -1)
    m.st(u[0] == 0)
    
    ################################################################################

    m.solve(ort)
    x_star = x.get()
    obj = m.get()
    return x_star, obj

Now you can solve the model to obtain the solution:

In [26]:
x_MTZ, obj_MTZ = TSP_MTZ(N, distance_mat[0:N,0:N])

Being solved by OR-Tools...
Solution status: 0
Running time: 0.6224s


### Obtain and visualize the routes given by the optimal solution

You can obtain the route via the following function:

In [27]:
seq_route_tsp, lon_route_tsp, lat_route_tsp = obtain_route(x_MTZ, lon[0:N], lat[0:N])
seq_route_tsp

array([ 1.,  2.,  3.,  4.,  5.,  6., 19., 18., 17., 16., 15., 13., 11.,
       12., 14.,  9.,  8.,  7., 10.,  1.])

In [28]:
lon_route_tsp

array([121.506511  , 121.5047659 , 121.50461254, 121.50461706,
       121.50434244, 121.50355982, 121.50111066, 121.50041452,
       121.49963304, 121.50102956, 121.50176195, 121.50270019,
       121.50404466, 121.50401728, 121.50457467, 121.50531689,
       121.50637397, 121.50911989, 121.50578096, 121.506511  ])

In [29]:
lat_route_tsp

array([31.29651182, 31.29504072, 31.2956814 , 31.29721625, 31.29800441,
       31.29754457, 31.29653253, 31.29650212, 31.29778307, 31.29927835,
       31.29901531, 31.29970307, 31.29908155, 31.29982957, 31.3002981 ,
       31.29974249, 31.30073522, 31.29959658, 31.29885704, 31.29651182])

In [30]:
FDmap = folium.Map(location=[lat[0], lon[0]], zoom_start=15,tiles='http://webrd02.is.autonavi.com/appmaptile?lang=zh_cn&size=1&scale=1&style=7&x={x}&y={y}&z={z}',attr='default') #高德地图
for i in range(N):
    folium.Marker(
        location=[lat_route_tsp[i], lon_route_tsp[i]],
        icon=BeautifyIcon(
            icon_shape='marker',
            number=int(i+1),
            spin=True,
            text_color='red',
            background_color="#FFF",
            inner_icon_style="font-size:12px;padding-top:-5px;"
        )
    ).add_to(FDmap)
FDmap

Now you can plot the (approximate) routes on the map:

In [31]:
for i in range(N):
    folium.PolyLine([[lat_route_tsp[i], lon_route_tsp[i]], [lat_route_tsp[i + 1], lon_route_tsp[i + 1]]]).add_to(FDmap)
FDmap

### Some issues with the solution:

1. Distance is an approximation;

2. Routes are not on the real roads.

You can use Gaode API to estimate the distance matrix and also use this package to draw the real routes for the example. You need to sign up and request an API Key. 

In [32]:
KEY='3bb4d4b617a629b3a641a284771920aa'

url= 'https://restapi.amap.com/v3/direction/walking?parameters'

dismat_walking = np.zeros((N,N))

for i in tqdm(range(N)):
    for j in range(N):
        ORI = '{},{}'.format(lon[i],lat[i])
        DES = '{},{}'.format(lon[j],lat[j])
        params = {'origin':ORI, 'destination':DES, 'key':KEY}
        response = requests.get(url, params = params)
        answer = response.json()
        print(answer)
        dismat_walking[i,j] = int(answer['route']['paths'][0]['distance'])
        time.sleep(0.5)

distance_mat = dismat_walking/1000

  0%|          | 0/19 [00:00<?, ?it/s]

{'status': '1', 'info': 'ok', 'infocode': '10000', 'count': '1', 'route': {'origin': '121.50651099789978,31.29651182318166', 'destination': '121.50651099789978,31.29651182318166', 'paths': [{'distance': '1', 'duration': '1', 'steps': [{'instruction': '向南步行1米到达目的地', 'orientation': '南', 'road': [], 'distance': '1', 'duration': '1', 'polyline': '121.50651,31.29651;121.50651,31.29651', 'action': [], 'assistant_action': '到达目的地', 'walk_type': '0'}]}]}}
{'status': '1', 'info': 'ok', 'infocode': '10000', 'count': '1', 'route': {'origin': '121.50651099789978,31.29651182318166', 'destination': '121.50476590018823,31.29504072065087', 'paths': [{'distance': '404', 'duration': '323', 'steps': [{'instruction': '向西南步行69米右转', 'orientation': '西南', 'road': [], 'distance': '69', 'duration': '55', 'polyline': '121.506463,31.296602;121.505903,31.296398;121.505903,31.296398;121.505803,31.296354', 'action': '右转', 'assistant_action': [], 'walk_type': '0'}, {'instruction': '沿国顺路向西北步行53米左转', 'orientation': '西北'

  5%|▌         | 1/19 [00:15<04:32, 15.15s/it]

{'status': '1', 'info': 'ok', 'infocode': '10000', 'count': '1', 'route': {'origin': '121.50476590018823,31.29504072065087', 'destination': '121.50651099789978,31.29651182318166', 'paths': [{'distance': '404', 'duration': '323', 'steps': [{'instruction': '向北步行37米右转', 'orientation': '北', 'road': [], 'distance': '37', 'duration': '30', 'polyline': '121.504883,31.295069;121.504766,31.295391', 'action': '右转', 'assistant_action': [], 'walk_type': '0'}, {'instruction': '向东步行34米左转', 'orientation': '东', 'road': [], 'distance': '34', 'duration': '27', 'polyline': '121.504761,31.295391;121.505009,31.295464;121.505009,31.295464;121.505113,31.295499', 'action': '左转', 'assistant_action': [], 'walk_type': '0'}, {'instruction': '沿国年路向西北步行115米右转', 'orientation': '西北', 'road': '国年路', 'distance': '115', 'duration': '92', 'polyline': '121.505113,31.295499;121.504935,31.295803;121.504935,31.295803;121.504709,31.296159;121.504709,31.296159;121.504601,31.296367;121.504592,31.296432', 'action': '右转', 'assist

 11%|█         | 2/19 [00:30<04:22, 15.46s/it]

{'status': '1', 'info': 'ok', 'infocode': '10000', 'count': '1', 'route': {'origin': '121.50461254465493,31.295681397520447', 'destination': '121.50651099789978,31.29651182318166', 'paths': [{'distance': '327', 'duration': '262', 'steps': [{'instruction': '向东步行32米左转', 'orientation': '东', 'road': [], 'distance': '32', 'duration': '26', 'polyline': '121.504601,31.295699;121.504805,31.29576;121.504805,31.29576;121.504931,31.295803', 'action': '左转', 'assistant_action': [], 'walk_type': '0'}, {'instruction': '沿国年路向西北步行77米右转', 'orientation': '西北', 'road': '国年路', 'distance': '77', 'duration': '62', 'polyline': '121.504931,31.295803;121.504709,31.296159;121.504709,31.296159;121.504601,31.296367;121.504592,31.296432', 'action': '右转', 'assistant_action': [], 'walk_type': '0'}, {'instruction': '沿中环路向东步行96米右转', 'orientation': '东', 'road': '中环路', 'distance': '96', 'duration': '77', 'polyline': '121.504588,31.296432;121.504653,31.296441;121.505135,31.296636;121.505525,31.296762', 'action': '右转', 'as

... (29 outputs left out)

{'status': '1', 'info': 'ok', 'infocode': '10000', 'count': '1', 'route': {'origin': '121.5004145197,31.29650211517389', 'destination': '121.50651099789978,31.29651182318166', 'paths': [{'distance': '751', 'duration': '601', 'steps': [{'instruction': '沿博学南路向东北步行193米右转', 'orientation': '东北', 'road': '博学南路', 'distance': '193', 'duration': '154', 'polyline': '121.500421,31.296489;121.500525,31.296549;121.500525,31.296549;121.501133,31.296853;121.501133,31.296853;121.502174,31.2974', 'action': '右转', 'assistant_action': [], 'walk_type': '0'}, {'instruction': '向东南步行68米左转', 'orientation': '东南', 'road': [], 'distance': '68', 'duration': '54', 'polyline': '121.502174,31.2974;121.502339,31.29714;121.502339,31.29714;121.502461,31.296936;121.502461,31.296936;121.502509,31.296853', 'action': '左转', 'assistant_action': [], 'walk_type': '0'}, {'instruction': '沿望道路向东步行260米右转', 'orientation': '东', 'road': '望道路', 'distance': '260', 'duration': '208', 'polyline': '121.502509,31.296849;121.503442,31.29717;

 95%|█████████▍| 18/19 [05:00<00:18, 18.44s/it]

{'status': '1', 'info': 'ok', 'infocode': '10000', 'count': '1', 'route': {'origin': '121.5011106597394,31.296532532053753', 'destination': '121.50651099789978,31.29651182318166', 'paths': [{'distance': '699', 'duration': '559', 'steps': [{'instruction': '向西北步行26米右转', 'orientation': '西北', 'road': [], 'distance': '26', 'duration': '21', 'polyline': '121.501272,31.296645;121.501137,31.296853', 'action': '右转', 'assistant_action': [], 'walk_type': '0'}, {'instruction': '沿博学南路向东北步行115米右转', 'orientation': '东北', 'road': '博学南路', 'distance': '115', 'duration': '92', 'polyline': '121.501133,31.296853;121.502174,31.2974', 'action': '右转', 'assistant_action': [], 'walk_type': '0'}, {'instruction': '向东南步行68米左转', 'orientation': '东南', 'road': [], 'distance': '68', 'duration': '54', 'polyline': '121.502174,31.2974;121.502339,31.29714;121.502339,31.29714;121.502461,31.296936;121.502461,31.296936;121.502509,31.296853', 'action': '左转', 'assistant_action': [], 'walk_type': '0'}, {'instruction': '沿望道路向东步行26

100%|██████████| 19/19 [05:19<00:00, 16.81s/it]


In [33]:
## plot the selected locations in a new map
TSPmap1 = folium.Map(location=[lat[0], lon[0]], zoom_start=16,tiles='http://webrd02.is.autonavi.com/appmaptile?lang=zh_cn&size=1&scale=1&style=7&x={x}&y={y}&z={z}',attr='default') #Stamen Terrain, OpenStreetMap
for i in range(N):
    folium.Marker(
        location=[lat[i], lon[i]],
        icon=BeautifyIcon(
            icon_shape='marker',
            number=int(i + 1),
            spin=True,
            text_color='red',
            background_color="#FFF",
            inner_icon_style="font-size:12px;padding-top:-5px;"
        )
    ).add_to(TSPmap1)
TSPmap1

In [34]:
########## Fill up your code here to obtain the optimal solution with updated distance matrix ##########
x_tsp, obj_tsp = TSP_MTZ(N, distance_mat[0:N,0:N])
########################################################################################################


Being solved by OR-Tools...
Solution status: 0
Running time: 8.3330s


In [35]:
seq_route_tsp, lon_route_tsp, lat_route_tsp = obtain_route(x_tsp, lon[0:N], lat[0:N])

In [36]:
seq_route_tsp

array([ 1.,  4.,  6.,  5., 11.,  9., 10.,  7.,  8., 14., 12., 13., 15.,
       16., 17., 18., 19.,  2.,  3.,  1.])

In [37]:
len(lon_route_tsp)

20

In [38]:
lon_route = lon_route_tsp
lat_route = lat_route_tsp

KEY='8a9532d5907371120264cd8dbf82e62a'

url= 'https://restapi.amap.com/v3/direction/walking?parameters'

for i in tqdm(range(N)):
    ORI = '{},{}'.format(lon_route[i],lat_route[i])
    DES = '{},{}'.format(lon_route[i+1],lat_route[i+1])
    params = {'origin':ORI, 'destination':DES, 'key':KEY}
    response = requests.get(url, params = params)
    answer = response.json()
    stpes = answer['route']['paths'][0]['steps']
    num_steps = len(stpes)
    time.sleep(0.5)
    for j in range(num_steps):
        coords_temp = stpes[j]['polyline']
        coords_list_str = coords_temp.split(';')
        #print(coords_list_str)
        coords_list = [list(map(float,coord_str.split(','))) for coord_str in coords_list_str]
        coords_list_rev = [[k[1],k[0]] for k in coords_list]
        #print(coords_list_rev)
        if j==0:
            folium.PolyLine([[lat_route[i],lon_route[i]],coords_list_rev[0]]).add_to(TSPmap1)
        if j==(num_steps-1):
            folium.PolyLine([coords_list_rev[-1],[lat_route[i+1],lon_route[i+1]]]).add_to(TSPmap1)
        folium.PolyLine(locations=coords_list_rev).add_to(TSPmap1)
folium.LayerControl().add_to(TSPmap1)
TSPmap1

100%|██████████| 19/19 [00:18<00:00,  1.03it/s]
